In [53]:
%reload_ext autoreload
%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
%reload_ext autoreload

In [58]:
# Reload the river.tree.hoeffding_tree_classifier module to get the updated methods
import importlib
import sys

# Remove cached modules
if 'river.tree.hoeffding_tree_classifier' in sys.modules:
    del sys.modules['river.tree.hoeffding_tree_classifier']
if 'river.tree' in sys.modules:
    del sys.modules['river.tree']

# Reimport
from river import tree
from river.tree import HoeffdingTreeClassifier
import time

# Verify the method exists
if hasattr(HoeffdingTreeClassifier, '_find_parent_and_index'):
    print("✅ Modules reloaded - _find_parent_and_index method available")
else:
    print("⚠️  Warning: _find_parent_and_index method not found")

✅ Modules reloaded - _find_parent_and_index method available


In [59]:
def split_callback(split_info):  
    import time
    print("\n" + "="*80)
    print("🌳 SPLIT EVENT DETECTED!")
    print("="*80)
    
    original_leaf = split_info['original_leaf']
    new_split_node = split_info['new_split_node']
    new_leaves = split_info['new_leaves']
    parent = split_info['parent']
    parent_branch = split_info['parent_branch']
    
    # Extract split node type and parameters
    split_node_type = type(new_split_node).__name__
    
    # Build branch parameters based on node type
    branch_params = {
        'feature': getattr(new_split_node, 'feature', None)
    }
    
    if hasattr(new_split_node, 'threshold'):
        branch_params['threshold'] = float(getattr(new_split_node, 'threshold'))
    
    if hasattr(new_split_node, 'value'):
        branch_params['value'] = getattr(new_split_node, 'value')
    
    if hasattr(new_split_node, 'radius'):
        branch_params['radius'] = float(getattr(new_split_node, 'radius'))
    
    if hasattr(new_split_node, '_mapping'):
        # For multiway splits, extract the mapping
        branch_params['feature_values'] = list(getattr(new_split_node, '_mapping', {}).keys())
    
    # Extract new leaf data
    new_leaves_data = []
    for idx, leaf in enumerate(new_leaves):
        leaf_data = {
            'node_id': getattr(leaf, 'node_id', None),
            'node_type': type(leaf).__name__,
            'depth': getattr(leaf, 'depth', 0),
            'branch_index': idx,
            'stats': {str(k): float(v) for k, v in getattr(leaf, 'stats', {}).items()},
            'total_weight': float(getattr(leaf, 'total_weight', 0)),
            'mc_correct_weight': float(getattr(leaf, '_mc_correct_weight', 0)),
            'nb_correct_weight': float(getattr(leaf, '_nb_correct_weight', 0)),
        }
        
        # Extract splitter data for each leaf
        if hasattr(leaf, 'splitters'):
            splitters_data = {}
            for feature_name, splitter in leaf.splitters.items():
                splitter_info = {
                    'type': type(splitter).__name__,
                    'feature_name': feature_name
                }
                
                # Check if Gaussian splitter
                if hasattr(splitter, '_att_dist_per_class'):
                    att_dist = splitter._att_dist_per_class
                    if att_dist:
                        first_val = next(iter(att_dist.values()))
                        
                        # Gaussian splitter
                        if hasattr(first_val, 'mu'):
                            gaussian_data = {}
                            distributions = {}
                            
                            for class_label, dist_obj in att_dist.items():
                                class_data = {
                                    'n_samples': float(dist_obj.n_samples) if hasattr(dist_obj, 'n_samples') else 0.0,
                                    'mu': float(dist_obj.mu) if hasattr(dist_obj, 'mu') else 0.0,
                                    'sigma': float(dist_obj.sigma) if hasattr(dist_obj, 'sigma') else 1.0
                                }
                                distributions[str(class_label)] = class_data
                            
                            gaussian_data['distributions'] = distributions
                            
                            if hasattr(splitter, '_min_per_class'):
                                gaussian_data['min_per_class'] = {
                                    str(k): float(v) for k, v in splitter._min_per_class.items()
                                }
                            
                            if hasattr(splitter, '_max_per_class'):
                                gaussian_data['max_per_class'] = {
                                    str(k): float(v) for k, v in splitter._max_per_class.items()
                                }
                            
                            splitter_info['gaussian_data'] = gaussian_data
                        
                        # Nominal splitter
                        elif isinstance(first_val, dict):
                            nominal_data = {
                                'class_distributions': {
                                    str(k): dict(v) for k, v in att_dist.items()
                                }
                            }
                            
                            if hasattr(splitter, '_att_values'):
                                nominal_data['unique_values'] = list(splitter._att_values)
                            
                            if hasattr(splitter, '_total_weight_observed'):
                                nominal_data['total_weight'] = float(splitter._total_weight_observed)
                            
                            splitter_info['nominal_data'] = nominal_data
                
                splitters_data[feature_name] = splitter_info
            
            leaf_data['splitters'] = splitters_data
        
        new_leaves_data.append(leaf_data)
    
    # Build complete split event payload
    split_event = {
        'event_type': 'split',
        'timestamp': __import__('time').time(),
        
        # Original leaf that was replaced
        'original_leaf_id': getattr(original_leaf, 'node_id', None),
        
        # New split node created
        'split_node': {
            'node_id': getattr(new_split_node, 'node_id', None),
            'node_type': split_node_type,
            'depth': getattr(new_split_node, 'depth', 0),
            'stats': {str(k): float(v) for k, v in getattr(new_split_node, 'stats', {}).items()},
            'branch_params': branch_params
        },
        
        # New leaf children
        'new_leaves': new_leaves_data,
        
        # Parent context
        'parent_context': {
            'parent_node_id': getattr(parent, 'node_id', None) if parent else None,
            'parent_branch': parent_branch
        }
    }
    
    print(f"✅ Split captured: {split_node_type} on feature '{branch_params.get('feature')}'")
    print(f"   Original leaf ID: {split_event['original_leaf_id']}")
    print(f"   New split node ID: {split_event['split_node']['node_id']}")
    print(f"   New leaves: {[leaf['node_id'] for leaf in new_leaves_data]}")
    print("="*80 + "\n")

    import os
    import time
    
    directory = 'json_data_lab/split_event'
    os.makedirs(directory, exist_ok=True)
    filename = f'json_data_lab/split_event/split_{int(time.time())}.json'
    with open(filename, 'w') as f:
        import json
        json.dump(split_event, f, indent=4)


In [60]:
def leaf_update_callback(update_info):
    import os
    import time
    # Delete existing directory and create a new one
    directory = 'json_data_lab/update_leaf'
    
    os.makedirs(directory, exist_ok=True)
    filename = f'json_data_lab/update_leaf/update_{time.time() * 10000}.json'
    with open(filename, 'w') as f:
        import json
        json.dump(update_info, f, indent=4)
    

In [77]:
update_tree = HoeffdingTreeClassifier(
            grace_period=200,
            leaf_prediction='nba',
            leaf_update_threshold=100,
            leaf_update_callback=leaf_update_callback,
            split_callback=split_callback
        )
from river.datasets import synth
dataset = synth.Agrawal(classification_function=0, seed=42)

import shutil
directory = 'json_data_lab'
shutil.rmtree(directory, ignore_errors=True)
for i, (x, y) in enumerate(dataset.take(700)):
    update_tree.learn_one(x, y)

   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive

🚪 GATE 2 TRIGGERED: Leaf 0 reached 100 instances
   Previous weight: 99.0 → Current weight: 100.0
   Threshold: 100 (multiple #1)
📦 CREATING UPDATE PAYLOAD:
   Node ID: 0 (LeafNaiveBayesAdaptive)
   Payload type: complete_node
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=94009.435, σ=41417.217), 0: 𝒩(μ=86880.423, σ=40443.395)}
      → Detected GAUSSIAN splitter for salary
      dist_obj for class 1: Gaussian
      dist_obj for class 0: Gaussian
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=19416.319, σ=28956.308), 0: 𝒩(μ=17035.556, σ=24406.304)}
      → Detected GAUSSIAN splitter for commission
      dist_obj for class 1: Gaussian
      dist_obj for class 0: Gaussian
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=49.889, σ=21.896), 0: 𝒩(μ=50.216, σ=6.566)}
      → Detected GAUSSIAN splitter for age
      dist_obj for class 1: Gaussian
 

In [82]:
inference_tree = HoeffdingTreeClassifier(
            grace_period=200,
            leaf_prediction='nba'
        )

for i, (x, y) in enumerate(dataset.take(10)):
    inference_tree.learn_one(x, y)

# read from json_data_lab/update_leaf the latest file
import os
import time
import json
directory = 'json_data_lab/update_leaf' 
files = os.listdir(directory)
files = [f for f in files if f.startswith('update_') and f.endswith('.json')]
files.sort(key=lambda x: os.path.getmtime(os.path.join(directory, x)), reverse=True)
latest_file = files[0]
with open(os.path.join(directory, latest_file), 'r') as f:
    update_info = json.load(f)
# update_info
inference_tree.apply_distributed_update(update_info)

   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive
❌ Node 1 not found in registry


False

In [86]:
# read from json_data_lab/split_event the latest file
import os
import time
import json
directory = 'json_data_lab/split_event' 
files = os.listdir(directory)
files = [f for f in files if f.startswith('split_') and f.endswith('.json')]
files.sort(key=lambda x: os.path.getmtime(os.path.join(directory, x)), reverse=True)
latest_file = files[0]
with open(os.path.join(directory, latest_file), 'r') as f:
    split_info = json.load(f)
split_info
inference_tree.apply_split_event(split_info)

📥 APPLYING SPLIT EVENT
Original leaf ID: 0
Split node ID: 3
Split node type: NumericBinaryBranch
Branch params: {'feature': 'age', 'threshold': 63.63636363636363}
New leaves: [1, 2]
   📊 Split data: {'event_type': 'split', 'timestamp': 1761214792.5846822, 'original_leaf_id': 0, 'split_node': {'node_id': 3, 'node_type': 'NumericBinaryBranch', 'depth': 0, 'stats': {'1': 385.0, '0': 215.0}, 'branch_params': {'feature': 'age', 'threshold': 63.63636363636363}}, 'new_leaves': [{'node_id': 1, 'node_type': 'LeafNaiveBayesAdaptive', 'depth': 1, 'branch_index': 0, 'stats': {'1': 281.2230429243434, '0': 215.0}, 'total_weight': 496.2230429243434, 'mc_correct_weight': 0.0, 'nb_correct_weight': 0.0, 'splitters': {}}, {'node_id': 2, 'node_type': 'LeafNaiveBayesAdaptive', 'depth': 1, 'branch_index': 1, 'stats': {'1': 103.77695707565658}, 'total_weight': 103.77695707565658, 'mc_correct_weight': 0.0, 'nb_correct_weight': 0.0, 'splitters': {}}], 'parent_context': {'parent_node_id': None, 'parent_branch':

True

In [87]:
# read from json_data_lab/update_leaf the latest file
import os
import time
import json
directory = 'json_data_lab/update_leaf' 
files = os.listdir(directory)
files = [f for f in files if f.startswith('update_') and f.endswith('.json')]
files.sort(key=lambda x: os.path.getmtime(os.path.join(directory, x)), reverse=True)
latest_file = files[0]
with open(os.path.join(directory, latest_file), 'r') as f:
    update_info = json.load(f)
# update_info
inference_tree.apply_distributed_update(update_info)

📡 APPLYING DISTRIBUTED UPDATE:
   Node ID: 1 (LeafNaiveBayesAdaptive)
   Update type: complete_node
   🔄 Applying complete node update...
   🍃 Applying leaf stats update...
      Current stats: {1: 281.2230429243434, 0: 215.0}
      New stats to sync: {'1': 281.2230429243434, '0': 219.0}
      Synchronized stats: {'1': 281.2230429243434, '0': 219.0}
      Expected weight: 500.2230429243434
      Actual weight after sync: 500.2230429243434
      ✅ Weight synchronized correctly
   🔀 Applying splitter data update...
      ⚠️ Feature salary not found in node splitters
      ⚠️ Feature commission not found in node splitters
      ⚠️ Feature age not found in node splitters
      ⚠️ Feature elevel not found in node splitters
      ⚠️ Feature car not found in node splitters
      ⚠️ Feature zipcode not found in node splitters
      ⚠️ Feature hvalue not found in node splitters
      ⚠️ Feature hyears not found in node splitters
      ⚠️ Feature loan not found in node splitters
      ✅ Splitter

True

In [79]:
update_tree._node_registry.get(1).stats

{1: 323.2230429243434, 0: 247.0}

In [83]:
inference_tree._node_registry.get(1).stats

AttributeError: 'NoneType' object has no attribute 'stats'

In [ ]:
class InferenceProcess:
    """Inference process that consumes updates from Kafka."""
    
    def __init__(self, instances):
        self.instances = instances
        self.tree = None
        self.consumer = None
        self.updates_received = 0
        self.running = False
        self.last_applied_iteration = 0  # Track last applied iteration for ordering
        
    def initialize_tree(self):
        """Initialize inference tree with both classes."""
        print("\n🎯 INFERENCE PROCESS STARTED")
        print("=" * 50)
        
        self.tree = HoeffdingTreeClassifier(grace_period=200, leaf_prediction='nba')
        
        print("🌱 Initializing tree with both classes...")
        
        # Find instances of both classes
        class_instances = {0: None, 1: None}
        for instance in self.instances[:20]:
            if instance['y'] in class_instances and class_instances[instance['y']] is None:
                class_instances[instance['y']] = instance
        
        # Learn from both classes
        for class_label, instance in class_instances.items():
            if instance:
                self.tree.learn_one(instance['x'], instance['y'])
                print(f"   ✅ Initialized with class {class_label}")
        
        print("✅ Inference tree initialized\n")
 
    def apply_update(self, update_data):
        """Apply Kafka update to inference tree."""
        node_id = update_data.get('node_id', 0)
        att_dist_per_class = update_data.get('_att_dist_per_class', {})
        stats_raw = update_data.get('stats', {})
        total_weight = update_data.get('total_weight', 0)
        mc_correct_weight = update_data.get('mc_correct_weight', 0)
        nb_correct_weight = update_data.get('nb_correct_weight', 0)
        
        # Convert stats keys from strings back to floats (JSON converts them to strings)
        stats = {float(k): v for k, v in stats_raw.items()}
        
        # Build the update payload
        update_payload = {
            'node_id': node_id,
            'update_type': 'complete_node',
            'timestamp': update_data.get('timestamp'),
            'data': {
                'leaf_stats': {
                    'stats': stats,
                    'total_weight': total_weight
                },
                'splitter_data': {
                    'splitters': {}
                },
                'naive_bayes_data': {
                    'mc_correct_weight': mc_correct_weight,
                    'nb_correct_weight': nb_correct_weight
                }
            }
        }
        
        # Convert distributions format
        for feature_name, class_distributions in att_dist_per_class.items():
            distributions_dict = {}
            for class_label, dist_params in class_distributions.items():
                distributions_dict[class_label] = {
                    'n_samples': dist_params.get('n_samples'),
                    'mu': dist_params.get('mu'),
                    'sigma': dist_params.get('sigma')
                }
            
            update_payload['data']['splitter_data']['splitters'][feature_name] = {
                'type': 'GaussianSplitter',
                'feature_name': feature_name,
                'gaussian_data': {
                    'distributions': distributions_dict
                }
            }
        
        # Apply the update
        if node_id in self.tree._node_registry:
            success = self.tree.apply_distributed_update(node_id, update_payload)
            print(f"   {'✅' if success else '❌'} Update applied to inference tree")
        else:
            print(f"   ⚠️ Node {node_id} not found in registry")

        
    def apply_split_event(self, split_data):
        """Apply a split event to reconstruct tree structure from distributed training.
        
        This method allows an inference process to reconstruct the tree structure
        by applying split events received from a distributed training process.
        
        Parameters
        ----------
        split_data : dict
            Dictionary containing split event information
        """
        print("=" * 70)
        print("📥 APPLYING SPLIT EVENT")
        print("=" * 70)
        
        original_leaf_id = split_data['original_leaf_id']
        split_node_info = split_data['split_node']
        new_leaves_info = split_data['new_leaves']
        
        print(f"Original leaf ID: {original_leaf_id}")
        print(f"Split node ID: {split_node_info['node_id']}")
        print(f"Split node type: {split_node_info['node_type']}")
        print(f"Branch params: {split_node_info['branch_params']}")
        print(f"New leaves: {[leaf['node_id'] for leaf in new_leaves_info]}")
        print(f"   📊 Split data keys: {list(split_data.keys())}")
        print()
        
        try:
            # Create new leaf children FIRST
            new_leaves = []
            for leaf_info in new_leaves_info:
                leaf = self._create_leaf_from_info(leaf_info)
                new_leaves.append(leaf)
                
            # Create the split node with children
            split_node = self._create_split_node(split_node_info, new_leaves)
            
            # Verify children are attached (for debugging)
            print(f"   Split node has {len(split_node.children)} children:")
            for i, child in enumerate(split_node.children):
                child_id = new_leaves_info[i]['node_id']
                print(f"      Child {i}: {type(child).__name__} (will be node_id={child_id})")
            
            # Replace the original leaf with the new split node
            if self.tree._root is None or original_leaf_id == 0:
                # Root split case
                self.tree._root = split_node
                print(f"✅ Replaced root with split node ID={split_node_info['node_id']}")
            else:
                # Non-root split: find parent and replace child
                parent_node, child_index = self.tree._find_parent_and_index(original_leaf_id)
                
                if parent_node is not None:
                    # Replace the child at the found index
                    parent_node.children[child_index] = split_node
                    print(f"✅ Replaced child at index {child_index} of parent node ID={getattr(parent_node, 'node_id', 'unknown')}")
                    print(f"   Original leaf ID: {original_leaf_id} → New split node ID: {split_node_info['node_id']}")
                else:
                    print(f"⚠️  Warning: Could not find parent for leaf ID {original_leaf_id}")
                    print(f"   This may indicate the tree structure is inconsistent")
            
            # Remove the original leaf from registry (it's being replaced)
            if original_leaf_id in self.tree._node_registry:
                old_leaf = self.tree._node_registry.pop(original_leaf_id)
                print(f"   🗑️  Removed original leaf ID={original_leaf_id} from registry")
            
            # Register new nodes in the model's node registry
            self.tree._register_node(split_node, split_node_info['node_id'])
            for leaf, leaf_info in zip(new_leaves, new_leaves_info):
                self.tree._register_node(leaf, leaf_info['node_id'])
            print(f"   Registry now contains: {list(self.tree._node_registry.keys())}")
            
            print(f"✅ Split event applied successfully")
            print(f"   Model state: {self.tree.n_nodes} nodes, height {self.tree.height}")
            print()

            # Check all nodes
            for node_id, node in self.tree._node_registry.items():
                print(f"   Node ID: {node_id}, Type: {type(node).__name__}")
                if hasattr(node, 'stats'):
                    print(f"      Stats: {node.stats}")
                    print(f"      Total weight: {node.total_weight}")
                    if hasattr(node, 'depth'):
                        print(f"      Depth: {node.depth}")
                if hasattr(node, 'feature'):
                    print(f"      Feature: {node.feature}")
                if hasattr(node, 'threshold'):
                    print(f"      Threshold: {node.threshold}")
                if hasattr(node, 'value'):
                    print(f"      Value: {node.value}")
                if hasattr(node, 'children'):
                    print(f"      Children count: {len(node.children)}")
                print()

                if hasattr(node, 'splitters') and node.splitters is not None:
                    from river.tree.splitter import GaussianSplitter
                    for feat, splitter in node.splitters.items():
                        print(f"      Splitter for feature '{feat}': {type(splitter).__name__}")
                        if isinstance(splitter, GaussianSplitter):
                            print(f"         Distributions: {splitter._att_dist_per_class}")
                            print(f"         Min per class: {splitter._min_per_class}")
                            print(f"         Max per class: {splitter._max_per_class}")
                    print()
            
            return True
            
        except Exception as e:
            print(f"❌ Error applying split event: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def _create_leaf_from_info(self, leaf_info):
        """Create a leaf node from leaf information."""
        from river.tree.nodes.htc_nodes import LeafMajorityClass, LeafNaiveBayes, LeafNaiveBayesAdaptive
        
        stats = {int(k): v for k, v in leaf_info['stats'].items()}
        depth = leaf_info.get('depth', 0)
        
        print(f"   Creating leaf: node_id={leaf_info.get('node_id')}, stats={stats}, depth={depth}")
        
        # Create leaf node using Naive Bayes Adaptive (matching tree's leaf_prediction)
        # CRITICAL: Use tree's splitter template, NOT None!
        leaf = LeafNaiveBayesAdaptive(stats=stats, depth=depth, splitter=self.tree.splitter)
        
        return leaf
    
    def _create_split_node(self, split_node_info, children):
        """Create a split node from split event data with children."""
        from river.tree.nodes.branch import NumericBinaryBranch, NominalBinaryBranch
        from river.tree.nodes.branch import NumericMultiwayBranch, NominalMultiwayBranch
        
        node_type = split_node_info['node_type']
        branch_params = split_node_info['branch_params']
        stats = split_node_info.get('stats', {})
        depth = split_node_info.get('depth', 0)
        
        # Convert stats from string keys to int
        stats_dict = {int(k): v for k, v in stats.items()} if stats else {}
        
        if node_type == 'NumericBinaryBranch':
            node = NumericBinaryBranch(
                stats=stats_dict,
                feature=branch_params['feature'],
                threshold=branch_params['threshold'],
                depth=depth,
                left=children[0],
                right=children[1]
            )
        elif node_type == 'NominalBinaryBranch':
            node = NominalBinaryBranch(
                stats=stats_dict,
                feature=branch_params['feature'],
                value=branch_params['value'],
                depth=depth,
                left=children[0],
                right=children[1]
            )
        elif node_type == 'NumericMultiwayBranch':
            node = NumericMultiwayBranch(
                stats=stats_dict,
                feature=branch_params['feature'],
                depth=depth,
                *children
            )
        elif node_type == 'NominalMultiwayBranch':
            node = NominalMultiwayBranch(
                stats=stats_dict,
                feature=branch_params['feature'],
                depth=depth,
                *children
            )
        else:
            raise ValueError(f"Unknown split node type: {node_type}")
        
        return node


In [43]:
from river.datasets import synth
dataset = synth.Agrawal(classification_function=0, seed=42)
instances = [{'x': x, 'y': y} for x, y in dataset.take(100)]
inference_process = InferenceProcess(instances=instances)
inference_process.initialize_tree()

# read json files from json_data_lab/update_leaf in order and apply updates to inference_process
import os
update_files = sorted(os.listdir('json_data_lab/update_leaf'), key=lambda x: int(x.split('_')[1].split('.')[0]))
for update_file in update_files:
    with open(os.path.join('json_data_lab/update_leaf', update_file), 'r') as f:
        import json
        update_data = json.load(f)
        # print(f"\nApplying update from file: {update_file}")
        # print(update_data)
        inference_process.apply_update(update_data) 


🎯 INFERENCE PROCESS STARTED
🌱 Initializing tree with both classes...
   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive
   ✅ Initialized with class 0
   ✅ Initialized with class 1
✅ Inference tree initialized

📡 APPLYING DISTRIBUTED UPDATE:
   Node ID: 0 (LeafNaiveBayesAdaptive)
   Update type: complete_node
   🔄 Applying complete node update...
   🍃 Applying leaf stats update...
      Current stats: {0: 1.0, 1: 1.0}
      New stats to sync: {1.0: 63.0, 0.0: 37.0}
      Synchronized stats: {1.0: 63.0, 0.0: 37.0}
      Expected weight: 100.0
      Actual weight after sync: 100.0
      ✅ Weight synchronized correctly
   🔀 Applying splitter data update...
      📊 Updating splitter for feature: salary
         🔢 Updating Gaussian splitter...
            🔄 Synchronizing Gaussian distribution for class 1
            ✅ Synced Gaussian: μ=94009.435377, σ=41417.217311, n=63.0
               (target: μ=94009.435377, σ=41417.217311, n=63.0)
         ✅ Updated class 1 distribution
        

In [54]:
training_process = TrainingProcess()
update_model = training_process.create_tree()

from river.datasets import synth

dataset = synth.Agrawal(classification_function=0, seed=42)
for i, (x, y) in enumerate(dataset.take(2000)):
    update_model.learn_one(x, y)

   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive

🚪 GATE 2 TRIGGERED: Leaf 0 reached 100 instances
   Previous weight: 99.0 → Current weight: 100.0
   Threshold: 100 (multiple #1)
📦 CREATING UPDATE PAYLOAD:
   Node ID: 0 (LeafNaiveBayesAdaptive)
   Payload type: complete_node
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=94009.435, σ=41417.217), 0: 𝒩(μ=86880.423, σ=40443.395)}
      → Detected GAUSSIAN splitter for salary
      dist_obj for class 1: Gaussian
      dist_obj for class 0: Gaussian
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=19416.319, σ=28956.308), 0: 𝒩(μ=17035.556, σ=24406.304)}
      → Detected GAUSSIAN splitter for commission
      dist_obj for class 1: Gaussian
      dist_obj for class 0: Gaussian
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=49.889, σ=21.896), 0: 𝒩(μ=50.216, σ=6.566)}
      → Detected GAUSSIAN splitter for age
      dist_obj for class 1: Gaussian
 

In [32]:
inference_process.tree._node_registry.get(0).splitters['salary']._att_dist_per_class

{0: 𝒩(μ=87821.726, σ=37317.823), 1: 𝒩(μ=86862.556, σ=38551.142)}

In [26]:
update_model._node_registry.get(0).splitters['salary']._att_dist_per_class

{1: 𝒩(μ=86570.484, σ=39128.275), 0: 𝒩(μ=86890.263, σ=37324.100)}

## Test Split Event Application

Let's apply split events from the training process to the inference tree.

In [44]:
# Read and apply split events from json_data_lab/verification
import os
import json

split_dir = 'json_data_lab/verification'
split_files = sorted([f for f in os.listdir(split_dir) if f.startswith('split_')], 
                     key=lambda x: int(x.split('_')[1].split('.')[0]))

print(f"Found {len(split_files)} split event(s)")
print(f"Files: {split_files}\n")

for split_file in split_files:
    print(f"\n{'='*70}")
    print(f"📂 Loading: {split_file}")
    print('='*70)
    
    with open(os.path.join(split_dir, split_file), 'r') as f:
        split_data = json.load(f)
    
    # Apply the split event
    success = inference_process.apply_split_event(split_data)
    
    if success:
        print(f"\n✅ Split event {split_file} applied successfully!")
    else:
        print(f"\n❌ Failed to apply split event {split_file}")
        break

Found 3 split event(s)
Files: ['split_1.json', 'split_2.json', 'split_3.json']


📂 Loading: split_1.json
📥 APPLYING SPLIT EVENT
Original leaf ID: 0
Split node ID: 3
Split node type: NumericBinaryBranch
Branch params: {'feature': 'age', 'threshold': 63.63636363636363}
New leaves: [1, 2]
   📊 Split data keys: ['event_type', 'timestamp', 'original_leaf_id', 'split_node', 'new_leaves', 'parent_context']

   Creating leaf: node_id=1, stats={1: 281.2230429243434, 0: 215.0}, depth=1
   Creating leaf: node_id=2, stats={1: 103.77695707565658}, depth=1
   Split node has 2 children:
      Child 0: LeafNaiveBayesAdaptive (will be node_id=1)
      Child 1: LeafNaiveBayesAdaptive (will be node_id=2)
✅ Replaced root with split node ID=3
   🗑️  Removed original leaf ID=0 from registry
   🏷️  REGISTERED NODE: ID=3, Type=NumericBinaryBranch
   🏷️  REGISTERED NODE: ID=1, Type=LeafNaiveBayesAdaptive
   🏷️  REGISTERED NODE: ID=2, Type=LeafNaiveBayesAdaptive
   Registry now contains: [3, 1, 2]
✅ Split event

In [45]:
# Verify the reconstructed tree structure
print("\n" + "="*70)
print("🔍 INFERENCE TREE STRUCTURE AFTER SPLIT EVENTS")
print("="*70)

print(f"\n📊 Tree Stats:")
print(f"   Total nodes: {inference_process.tree.n_nodes}")
print(f"   Tree height: {inference_process.tree.height}")
print(f"   Active leaves: {inference_process.tree.n_active_leaves}")
print(f"   Inactive leaves: {inference_process.tree.n_inactive_leaves}")
print(f"   Registry size: {len(inference_process.tree._node_registry)}")

print(f"\n🗂️  Node Registry:")
for node_id in sorted(inference_process.tree._node_registry.keys()):
    node = inference_process.tree._node_registry[node_id]
    node_type = type(node).__name__
    
    if hasattr(node, 'feature'):  # Branch node
        feature = node.feature
        threshold = getattr(node, 'threshold', None)
        print(f"   Node {node_id}: {node_type} | Feature: {feature} | Threshold: {threshold}")
    else:  # Leaf node
        stats = getattr(node, 'stats', {})
        total_weight = getattr(node, 'total_weight', 0)
        print(f"   Node {node_id}: {node_type} | Stats: {stats} | Weight: {total_weight:.1f}")

print("\n" + "="*70)


🔍 INFERENCE TREE STRUCTURE AFTER SPLIT EVENTS

📊 Tree Stats:
   Total nodes: 7
   Tree height: 4
   Active leaves: 1
   Inactive leaves: 0
   Registry size: 7

🗂️  Node Registry:
   Node 2: LeafNaiveBayesAdaptive | Stats: {1: 103.77695707565658} | Weight: 103.8
   Node 3: NumericBinaryBranch | Feature: age | Threshold: 63.63636363636363
   Node 4: LeafNaiveBayesAdaptive | Stats: {1: 66.73616799628577} | Weight: 66.7
   Node 6: NumericBinaryBranch | Feature: age | Threshold: 39.54545454545455
   Node 7: LeafNaiveBayesAdaptive | Stats: {0: 154.17448112835524} | Weight: 154.2
   Node 8: LeafNaiveBayesAdaptive | Stats: {0: 6.825518871644761, 1: 39.0} | Weight: 45.8
   Node 9: NumericBinaryBranch | Feature: age | Threshold: 58.81818181818181



In [46]:
# Test predictions on reconstructed tree
print("\n" + "="*70)
print("🔮 TESTING PREDICTIONS ON RECONSTRUCTED TREE")
print("="*70)

# Use test instances
from river.datasets import synth
test_dataset = synth.Agrawal(classification_function=0, seed=999)
test_samples = list(test_dataset.take(10))

correct = 0
for i, (x, y_true) in enumerate(test_samples, 1):
    y_pred = inference_process.tree.predict_one(x)
    y_proba = inference_process.tree.predict_proba_one(x)
    
    is_correct = (y_pred == y_true)
    correct += is_correct
    
    print(f"\n{'✅' if is_correct else '❌'} Sample {i}:")
    print(f"   Age: {x['age']:.1f}, Salary: {x['salary']:.0f}")
    print(f"   True: {y_true}, Predicted: {y_pred}")
    print(f"   Probabilities: {y_proba}")

accuracy = correct / len(test_samples)
print(f"\n📊 Accuracy: {accuracy:.1%} ({correct}/{len(test_samples)})")
print("="*70)


🔮 TESTING PREDICTIONS ON RECONSTRUCTED TREE

✅ Sample 1:
   Age: 25.0, Salary: 121575
   True: 1, Predicted: 1
   Probabilities: {0: 0.0, 1: 1.0}

✅ Sample 2:
   Age: 61.0, Salary: 104161
   True: 1, Predicted: 1
   Probabilities: {0: 0.1489458066096913, 1: 0.8510541933903086}

✅ Sample 3:
   Age: 45.0, Salary: 121324
   True: 0, Predicted: 0
   Probabilities: {0: 1.0, 1: 0.0}

✅ Sample 4:
   Age: 33.0, Salary: 133498
   True: 1, Predicted: 1
   Probabilities: {0: 0.0, 1: 1.0}

✅ Sample 5:
   Age: 49.0, Salary: 140078
   True: 0, Predicted: 0
   Probabilities: {0: 1.0, 1: 0.0}

✅ Sample 6:
   Age: 66.0, Salary: 148717
   True: 1, Predicted: 1
   Probabilities: {0: 0.0, 1: 1.0}

✅ Sample 7:
   Age: 49.0, Salary: 120499
   True: 0, Predicted: 0
   Probabilities: {0: 1.0, 1: 0.0}

✅ Sample 8:
   Age: 33.0, Salary: 123429
   True: 1, Predicted: 1
   Probabilities: {0: 0.0, 1: 1.0}

✅ Sample 9:
   Age: 77.0, Salary: 131720
   True: 1, Predicted: 1
   Probabilities: {0: 0.0, 1: 1.0}

✅ Samp

## Test Updated Leaf Update Callback

Test that `create_update_payload` is being used in the leaf update callback.

In [55]:
# Test that the new callback includes update_payload
captured_callback_data = {}

def test_callback(info):
    """Capture the callback data to verify it includes update_payload"""
    global captured_callback_data
    captured_callback_data = info
    print(f"✅ Callback received for node {info['node_id']}")
    print(f"   Keys in info: {list(info.keys())}")
    if 'update_payload' in info:
        print(f"   ✅ update_payload is present!")
        print(f"   Update payload keys: {list(info['update_payload'].keys())}")
    else:
        print(f"   ❌ update_payload is missing!")

# Create a test tree with the new callback
test_tree = HoeffdingTreeClassifier(
    grace_period=200,
    leaf_prediction='nba',
    leaf_update_threshold=50,  # Trigger after 50 samples
    leaf_update_callback=test_callback
)

# Train on some samples to trigger the callback
from river.datasets import synth
dataset = synth.Agrawal(classification_function=0, seed=42)

print("Training to trigger leaf update callback...\n")
for i, (x, y) in enumerate(dataset.take(100), 1):
    test_tree.learn_one(x, y)
    if i == 100:
        print(f"\nCompleted {i} samples")

print("\n" + "="*70)
print("CAPTURED CALLBACK DATA STRUCTURE:")
print("="*70)
for key in captured_callback_data.keys():
    if key == 'update_payload':
        print(f"\n✅ {key}:")
        payload = captured_callback_data[key]
        print(f"   - node_id: {payload.get('node_id')}")
        print(f"   - update_type: {payload.get('update_type')}")
        print(f"   - timestamp: {payload.get('timestamp')}")
        print(f"   - data sections: {list(payload.get('data', {}).keys())}")
    elif key == 'leaf_data':
        print(f"\n📊 {key}:")
        leaf_data = captured_callback_data[key]
        print(f"   - Keys: {list(leaf_data.keys())}")
    else:
        print(f"\n{key}: {captured_callback_data.get(key)}")

Training to trigger leaf update callback...

   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive

🚪 GATE 2 TRIGGERED: Leaf 0 reached 50 instances
   Previous weight: 49.0 → Current weight: 50.0
   Threshold: 50 (multiple #1)
📦 CREATING UPDATE PAYLOAD:
   Node ID: 0 (LeafNaiveBayesAdaptive)
   Payload type: complete_node
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=103573.293, σ=40655.862), 0: 𝒩(μ=70765.429, σ=34496.762)}
      → Detected GAUSSIAN splitter for salary
      dist_obj for class 1: Gaussian
      dist_obj for class 0: Gaussian
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=17291.427, σ=28566.298), 0: 𝒩(μ=24922.723, σ=28354.552)}
      → Detected GAUSSIAN splitter for commission
      dist_obj for class 1: Gaussian
      dist_obj for class 0: Gaussian
      splitter: GaussianSplitter
      splitter._att_dist_per_class: {1: 𝒩(μ=48.971, σ=21.694), 0: 𝒩(μ=49.937, σ=6.255)}
      → Detected GAUSSIAN splitter for

In [56]:
# Test that update_payload can be used directly with apply_distributed_update
print("="*70)
print("TESTING DIRECT UPDATE APPLICATION")
print("="*70)

# Create a new tree (receiver)
receiver_tree = HoeffdingTreeClassifier(grace_period=200, leaf_prediction='nba')

# Train it on a few samples to get a leaf node
for i, (x, y) in enumerate(synth.Agrawal(classification_function=0, seed=999).take(10)):
    receiver_tree.learn_one(x, y)

print(f"\n📊 Receiver tree initial state:")
print(f"   Total nodes: {len(receiver_tree._node_registry)}")
print(f"   Root node weight: {receiver_tree._root.total_weight}")

# Apply the update payload from the captured callback
print(f"\n📤 Applying update from captured callback...")
print(f"   Node ID: {captured_callback_data['update_payload']['node_id']}")
print(f"   Update type: {captured_callback_data['update_payload']['update_type']}")

# Apply the update
result = receiver_tree.apply_distributed_update(
    node_id=captured_callback_data['node_id'],
    update_payload=captured_callback_data['update_payload']
)

print(f"\n✅ Update applied: {result}")
print(f"\n📊 Receiver tree after update:")
print(f"   Total nodes: {len(receiver_tree._node_registry)}")
print(f"   Root node weight: {receiver_tree._root.total_weight}")

# Verify the update was applied by checking stats
if hasattr(receiver_tree._root, 'stats'):
    print(f"   Root stats: {dict(receiver_tree._root.stats)}")
    
print("\n" + "="*70)
print("✅ SUCCESS: update_payload can be used directly with apply_distributed_update!")
print("="*70)

TESTING DIRECT UPDATE APPLICATION
   🏷️  REGISTERED NODE: ID=0, Type=LeafNaiveBayesAdaptive

📊 Receiver tree initial state:
   Total nodes: 1
   Root node weight: 10.0

📤 Applying update from captured callback...
   Node ID: 0
   Update type: complete_node
📡 APPLYING DISTRIBUTED UPDATE:
   Node ID: 0 (LeafNaiveBayesAdaptive)
   Update type: complete_node
   🔄 Applying complete node update...
   🍃 Applying leaf stats update...
      Current stats: {1: 7.0, 0: 3.0}
      New stats to sync: {1: 63.0, 0: 37.0}
      Synchronized stats: {1: 63.0, 0: 37.0}
      Expected weight: 100.0
      Actual weight after sync: 100.0
      ✅ Weight synchronized correctly
   🔀 Applying splitter data update...
      📊 Updating splitter for feature: salary
         🔢 Updating Gaussian splitter...
            🔄 Synchronizing Gaussian distribution for class 1
            ✅ Synced Gaussian: μ=94009.435377, σ=41417.217311, n=63.0
               (target: μ=94009.435377, σ=41417.217311, n=63.0)
         ✅ Update

## ✅ Refactoring Complete: Unified Payload Creation

### Summary
The `_check_leaf_update_gate` method has been successfully refactored to use `create_update_payload` instead of manual data extraction.

### Key Changes
1. **Eliminated Code Duplication**: Removed ~20 lines of duplicate data extraction logic
2. **Single Source of Truth**: Now uses `create_update_payload` for consistent payload creation
3. **Enhanced Callback**: Added `update_payload` field to callback info dictionary

### Callback Info Structure
The leaf update callback now receives:
```python
{
    'gate_type': 'leaf_update',
    'node_id': 0,
    'node_type': 'LeafNaiveBayesAdaptive',
    'previous_weight': 99.0,
    'current_weight': 100.0,
    'threshold': 50,
    'multiple_reached': 2,
    'total_instances_reached': 100,
    'leaf_data': {         # Backward compatible - existing format
        'depth': ...,
        'stats': ...,
        'total_weight': ...,
        'is_active': ...,
        'naive_bayes': ...,
        'splitters': ...
    },
    'timestamp': 1761198547.8207982,
    'tree_id': 125652833934400,
    'update_payload': {    # NEW - Can be used directly with apply_distributed_update
        'node_id': 0,
        'update_type': 'complete_node',
        'timestamp': 1761198547.8207982,
        'source_tree_id': 125652833934400,
        'data': {
            'leaf_stats': {...},
            'splitter_data': {...},
            'naive_bayes_data': {...}
        }
    }
}
```

### Usage Patterns
**Option 1: Direct Application (Recommended)**
```python
def leaf_update_callback(info):
    # Use the pre-built update_payload directly
    receiver_tree.apply_distributed_update(
        node_id=info['node_id'],
        update_payload=info['update_payload']
    )
```

**Option 2: Custom Processing (Backward Compatible)**
```python
def leaf_update_callback(info):
    # Use leaf_data for custom logic
    leaf_data = info['leaf_data']
    # ... custom processing ...
```

### Benefits
- ✅ **Consistency**: Same payload format across split and leaf update callbacks
- ✅ **Maintainability**: Single method to update when payload format changes
- ✅ **Efficiency**: Pre-built payload ready for direct application
- ✅ **Flexibility**: Both `update_payload` and `leaf_data` available for different use cases
- ✅ **Backward Compatible**: Existing code using `leaf_data` continues to work